# G12 serving check: vLLM OpenAI endpoint with the G10 chat template and the g10 parsers

Kaggle: Accelerator **GPU T4 x2**, Internet **on**. Colab: any GPU runtime (L4 or A100 computes in bf16).

1. Smoke first: `SMOKE = True`, run all cells interactively (about 15 min: install, tests, download, 10 prompts).
2. Real run: `SMOKE = False`, then **Save Version → Save & Run All (Commit)**. Download `serve_check_results.zip`
   from that version's Output tab. Copy its `results/` folder into `teams/12-quantization/`. Leave `logs/` out of
   the repo.

Steps:
1. Clone the branch and install vLLM 0.33.0 in its own environment (the version in `gpu_path/results/environment.json`).
2. Run the parser tests, including the ones that go through vLLM's own parser code.
3. Download the model.
4. Run `serving/serve_check.py`. It starts `vllm serve` with the G10 template and the g10 reasoning and tool
   parsers, sends the 100 fixed prompts (and a streamed copy of 20), and checks that reasoning and tool calls land
   in `message.reasoning` / `message.tool_calls`.
5. Zip the report.

The zip is written even when the check fails, so the log is never lost. The last cell then stops with an error.

In [1]:
# ---- settings ----
SMOKE = False                 # True: 10 prompts, 5 streamed (checks every step); False: all 100 prompts, 20 streamed
RUN_BY = "SomaKorada07"
REPO_URL = "https://github.com/SomaKorada07/era-v5-lab.git"
BRANCH = "g12-week3-soma"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"      # stand-in (Apache-2.0) until a dry-run-1 checkpoint is exported
REVISION = None              # None = latest; the commit used is written to environment.json
LABEL = "Qwen3-4B-Instruct-2507 (stand-in)"
FORMAT_HINT = "on"           # "on": a system message teaches the G10 markers; "off" for a model trained on them
MAX_MODEL_LEN = 4096
MAX_TOKENS = 1024
USD_PER_GPU_HOUR = 0.0       # Kaggle / Colab free tier
PKGS = ["vllm==0.33.0", "pytest", "jinja2"]

In [2]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working" if KAGGLE else "/content")
TMP = Path("/kaggle/tmp" if KAGGLE else "/content/tmp")      # model weights, outside the Kaggle output folder
OUT = WORK / ("serve_check_smoke" if SMOKE else "serve_check_out")
for d in (TMP, OUT): d.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True, **kw):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True, **kw)
    if check and p.returncode: raise SystemExit(f"failed ({p.returncode}): {cmd}")
    return p.returncode

REPO = TMP / "era-v5-lab"
if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
SV = REPO / "teams/12-quantization/serving"
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

q = "nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader"
gpus = [l for l in subprocess.run(q, shell=True, capture_output=True, text=True).stdout.splitlines() if l.strip()]
assert gpus, "no GPU: set the accelerator to GPU T4 x2"
N_GPUS = len(gpus)
TP = 2 if N_GPUS >= 2 else 1
cc = min(float(g.split(",")[-1]) for g in gpus)
DTYPE = "bfloat16" if cc >= 8.0 else "half"                   # T4 (7.5) and P100 have no bf16
env = {"started": time.strftime("%Y-%m-%dT%H:%M:%S"), "platform": "kaggle" if KAGGLE else "colab",
       "run_by": RUN_BY, "repo": REPO_URL, "branch": BRANCH, "commit": commit, "gpu": gpus, "tp": TP, "dtype": DTYPE,
       "settings": {k: globals()[k] for k in ("SMOKE", "MODEL", "REVISION", "LABEL", "FORMAT_HINT", "MAX_MODEL_LEN",
                                               "MAX_TOKENS", "PKGS")}}
print(json.dumps(env, indent=1))

$ git clone --depth 1 --branch g12-week3-soma https://github.com/SomaKorada07/era-v5-lab.git /kaggle/tmp/era-v5-lab


Cloning into '/kaggle/tmp/era-v5-lab'...
Updating files: 100% (2597/2597), done.


{
 "started": "2026-11-15T14:47:26",
 "platform": "kaggle",
 "run_by": "SomaKorada07",
 "repo": "https://github.com/SomaKorada07/era-v5-lab.git",
 "branch": "g12-week3-soma",
 "commit": "ba5fdc0ea9c399d0345ea7ff23f6791c15ac3bdc",
 "gpu": [
  "Tesla T4, 15360 MiB, 7.5",
  "Tesla T4, 15360 MiB, 7.5"
 ],
 "tp": 2,
 "dtype": "half",
 "settings": {
  "SMOKE": false,
  "MODEL": "Qwen/Qwen3-4B-Instruct-2507",
  "REVISION": null,
  "LABEL": "Qwen3-4B-Instruct-2507 (stand-in)",
  "FORMAT_HINT": "on",
  "MAX_MODEL_LEN": 4096,
  "MAX_TOKENS": 1024,
  "PKGS": [
   "vllm==0.33.0",
   "pytest",
   "jinja2"
  ]
 }
}


In [3]:
# vLLM in its own environment: it pins its own torch and transformers
sh("pip -q install uv")
VENV = TMP / "venv-vllm"
if not (VENV / "bin/python").exists():
    sh(f"uv venv -q --python 3.12 {VENV}")
    sh(f"uv pip install -q --python {VENV}/bin/python " + " ".join(f"'{p}'" for p in PKGS))
PY = VENV / "bin/python"
code_ = "import importlib.metadata as m; print({k: m.version(k) for k in 'vllm torch transformers'.split()})"
env["packages"] = subprocess.run([str(PY), "-c", code_], capture_output=True, text=True, check=True).stdout.strip()
print(env["packages"])

$ pip -q install uv
$ uv venv -q --python 3.12 /kaggle/tmp/venv-vllm
$ uv pip install -q --python /kaggle/tmp/venv-vllm/bin/python 'vllm==0.33.0' 'pytest' 'jinja2'
{'vllm': '0.33.0', 'torch': '2.14.0', 'transformers': '5.20.2'}


In [4]:
# parser tests: template round trip, streaming, and the parsers inside vLLM's own DelegatingParser; none may skip
sh(f"{PY} -m pytest {SV} -q -rs -p no:cacheprovider 2>&1 | tee {OUT}/tests.log", executable="/bin/bash")
log = open(f"{OUT}/tests.log").read()
assert " failed" not in log and "skipped" not in log, "a parser test failed or skipped: see tests.log"

$ /kaggle/tmp/venv-vllm/bin/python -m pytest /kaggle/tmp/era-v5-lab/teams/12-quantization/serving -q -rs -p no:cacheprovider 2>&1 | tee /kaggle/working/serve_check_out/tests.log
................................................................         [100%]
64 passed in 25.88s


In [5]:
MODEL_DIR = TMP / "models" / MODEL.split("/")[-1]
if not (MODEL_DIR / "config.json").exists():
    sh(f"{PY} -c \"from huggingface_hub import snapshot_download as s; "
       f"s('{MODEL}', revision={REVISION!r}, local_dir='{MODEL_DIR}', "
       f"allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])\"")
info = subprocess.run([str(PY), "-c", f"from huggingface_hub import model_info as i; print(i('{MODEL}', "
                       f"revision={REVISION!r}).sha)"], capture_output=True, text=True)
env["model_commit"] = info.stdout.strip() or None
print("model", MODEL, "commit", env["model_commit"], "size", round(sum(f.stat().st_size for f in MODEL_DIR.rglob("*")
      if f.is_file()) / 1e9, 2), "GB")

$ /kaggle/tmp/venv-vllm/bin/python -c "from huggingface_hub import snapshot_download as s; s('Qwen/Qwen3-4B-Instruct-2507', revision=None, local_dir='/kaggle/tmp/models/Qwen3-4B-Instruct-2507', allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])"


Fetching 10 files: 100%|██████████| 10/10 [00:32<00:00,  3.25s/it]


model Qwen/Qwen3-4B-Instruct-2507 commit cdbee75f17c01a7cc42f958dc650907174af0554 size 8.06 GB


In [6]:
# the check itself; it starts and stops the vLLM server
smoke = "--limit 10 --stream-check 5" if SMOKE else ""
t = time.time()
rc = sh(f"set -o pipefail; {PY} {SV}/serve_check.py --model {MODEL_DIR} --label '{LABEL}' --out {OUT} "
        f"--format-hint {FORMAT_HINT} --max-tokens {MAX_TOKENS} --n-gpus {N_GPUS} --usd-per-gpu-hour {USD_PER_GPU_HOUR} "
        f"--vllm-arg=--dtype={DTYPE} --vllm-arg=--max-model-len={MAX_MODEL_LEN} --vllm-arg=--tensor-parallel-size={TP} "
        f"{smoke} 2>&1 | tee {OUT}/serve_check.log", check=False, executable="/bin/bash")
env["serve_check_exit"] = rc
env["wall_clock_s"] = round(time.time() - t)

$ set -o pipefail; /kaggle/tmp/venv-vllm/bin/python /kaggle/tmp/era-v5-lab/teams/12-quantization/serving/serve_check.py --model /kaggle/tmp/models/Qwen3-4B-Instruct-2507 --label 'Qwen3-4B-Instruct-2507 (stand-in)' --out /kaggle/working/serve_check_out --format-hint on --max-tokens 1024 --n-gpus 2 --usd-per-gpu-hour 0.0 --vllm-arg=--dtype=half --vllm-arg=--max-model-len=4096 --vllm-arg=--tensor-parallel-size=2  2>&1 | tee /kaggle/working/serve_check_out/serve_check.log
W1115 14:53:15.303000 226 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 14:53:15.304000 226 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1115 14:53:15.306000 226 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
launch: /kaggle/tmp/venv-vllm/bin/vllm serve /kaggle/tmp/models/Qwen3-4B-Instruct-2507 --served-model-name Qwen3-4B-Instruct-2507 --host 127.0.0.1 --port 8

In [7]:
# results/serve_check.md + results/serve_check/{summary.json, serve_check.jsonl, environment.json}; logs/ stays out of git
env["finished"] = time.strftime("%Y-%m-%dT%H:%M:%S")
PKG = WORK / "serve_check_pkg"
shutil.rmtree(PKG, ignore_errors=True)
(PKG / "results/serve_check").mkdir(parents=True)
(PKG / "logs").mkdir()
suffix = "_smoke" if SMOKE else ""
if (OUT / "serve_check.md").exists():
    shutil.copy(OUT / "serve_check.md", PKG / f"results/serve_check{suffix}.md")
for f in ("summary.json", "serve_check.jsonl"):
    if (OUT / f).exists():
        shutil.copy(OUT / f, PKG / "results/serve_check" / f.replace(".", f"{suffix}.", 1))
(PKG / f"results/serve_check/environment{suffix}.json").write_text(json.dumps(env, indent=1) + "\n")
for f in ("vllm_server.log", "serve_check.log", "tests.log"):
    if (OUT / f).exists():
        shutil.copy(OUT / f, PKG / "logs" / f)
zip_path = shutil.make_archive(str(WORK / f"serve_check_results{suffix}"), "zip", PKG)
print("results:", zip_path)
if (OUT / "serve_check.md").exists():
    print(open(OUT / "serve_check.md").read())
if rc != 0:
    print(open(OUT / "vllm_server.log").read()[-3000:] if (OUT / "vllm_server.log").exists() else "")
    raise SystemExit(f"serve_check exit code {rc}: see the report above and logs/ in the zip")

results: /kaggle/working/serve_check_results.zip
# Serving check: Qwen3-4B-Instruct-2507 (stand-in)

2026-11-15 14:58 UTC · vLLM 0.33.0 · GPU Tesla T4, 15360 MiB, 7.5; Tesla T4, 15360 MiB, 7.5 · model `/kaggle/tmp/models/Qwen3-4B-Instruct-2507`

## Result

- **Requests:** 100/100 returned HTTP 200.
- **Fields (the serving check): PASS.** 98/100 answers have no chat marker in `content`, `reasoning` or tool arguments. Every other answer was replayed (raw text regenerated, parsed offline by g10_parsers): model broke the G10 layout and the parser placed it as specified: 2; parser fault or not explained: 0.
- **Reasoning in `message.reasoning`:** 25/25 reason prompts.
- **Tool calls in `message.tool_calls`:** 28/30 tool prompts call the expected tool with valid JSON arguments.
- **No tool call when none is needed:** 20/20 (tool_result + no_tool prompts).
- **Streaming matches non-streaming:** 19/20 streamed prompts give the same reasoning, content and tool calls.
- **Model fell back to its 